## Домашняя мини-задача


1) Возьмите **русский роман** (любой .txt) и повторите блок **Zipf + длинный хвост**.
2) Сравните предобработки: lowercasing, удаление пунктуации, замена "ё"→"е", лемматизация (если есть инструменты).
3) Добавьте 2 новых интента и правила (Matcher/regex).


## 1) Загрузка данных (Yandex disk)

In [ ]:
import re
from collections import Counter
import requests

In [ ]:
YANDEX_URL = "https://disk.yandex.ru/d/jgpiPGPAZ6W7rg"

In [ ]:
api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"
params = {"public_key": YANDEX_URL}

response = requests.get(api_url, params=params, timeout=30)
response

In [ ]:
data = response.json()
data

In [ ]:
data.get('href')

In [ ]:
raw = requests.get(data.get('href'),timeout=30).content
raw

In [ ]:
text = raw.decode("utf-8")
print(text)

## 2) Токенизация, types/tokens, частоты

In [ ]:

# Простая токенизация для частот: слова из латинских букв + апострофы
tokens = re.findall(r"[А-Яа-я]+(?:'[А-Яа-я]+)?", text.lower())
cnt = Counter(tokens)

N_tokens = sum(cnt.values())  # tokens = все употребления
V_types = len(cnt)            # types  = уникальные слова

print(f"Tokens (всего словоупотреблений): {N_tokens:,}")
print(f"Types  (уникальных слов):        {V_types:,}")

# Топ-20
top20 = cnt.most_common(20)
top20[:10]


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Zipf данные
freqs_sorted = np.array(sorted(cnt.values(), reverse=True))
ranks = np.arange(1, len(freqs_sorted) + 1)

# 3.1 Zipf log-log
plt.figure(figsize=(7,5))
plt.loglog(ranks, freqs_sorted, marker=".", linestyle="none")
plt.title("Pride and Prejudice: закон Ципфа (ранг–частота)")
plt.xlabel("Ранг слова (log)")
plt.ylabel("Частота (log)")
plt.tight_layout()
plt.show()

# 3.2 Топ-20
df_top20 = pd.DataFrame(top20, columns=["word","count"])
plt.figure(figsize=(10,4))
plt.bar(df_top20["word"], df_top20["count"])
plt.title("Топ-20 слов по частоте")
plt.xlabel("Слово")
plt.ylabel("Частота")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 3.3 Длинный хвост: <= 3
tail_words = [w for w, c in cnt.items() if c <= 3]
tail_types = len(tail_words)
tail_tokens = sum(cnt[w] for w in tail_words)

print(f"Хвост (c<=3): {tail_types} типов = {tail_types/V_types:.1%} словаря")
print(f"Хвост (c<=3): {tail_tokens} токенов = {tail_tokens/N_tokens:.1%} текста")

plt.figure(figsize=(7,4))
plt.bar(["Доля ТИПОВ\n(c ≤ 3)", "Доля ТОКЕНОВ\n(c ≤ 3)"],
        [tail_types/V_types, tail_tokens/N_tokens])
plt.ylim(0, 1)
plt.title("«Длинный хвост» (c ≤ 3)")
plt.ylabel("Доля")
plt.tight_layout()
plt.show()

# 3.4 (Опционально) 10 случайных hapax (c=1)
hapax = [w for w, c in cnt.items() if c == 1]
rng = np.random.default_rng(0)
sample10 = list(rng.choice(hapax, size=10, replace=False))
sample10


## 4) Классический NLP-пайплайн в spaCy

In [ ]:
import spacy
print(spacy.__version__)

In [ ]:
!python -m spacy download ru_core_news_sm

In [ ]:
nlp = spacy.load("ru_core_news_sm")

In [ ]:
doc = nlp("Матушка была ещё мною брюхата, как уже я был записан в Семёновский полк сержантом, по милости майора гвардии князя Б., близкого нашего родственника. Если бы паче всякого чаяния матушка родила дочь, то батюшка объявил бы куда следовало о смерти неявившегося сержанта, и дело тем бы и кончилось")
print("SENTENCES:", [s.text for s in doc.sents])
print("TOKENS:", [t.text for t in doc])
print("POS:", [(t.text, t.pos_) for t in doc])
print("DEP:", [(t.text, t.dep_, t.head.text) for t in doc])

print("\nNER entities:")
for ent in doc.ents:
    print(ent.text, ent.label_)

for t in doc:
    print(f"{t.i:2d}  {t.text!r:12}  is_alpha={t.is_alpha}  is_punct={t.is_punct}")


### Применим пайплайн к реальному фрагменту из книги

In [ ]:

# Возьмём небольшой фрагмент (чтобы быстро работало)
snippet = " ".join(text.split()[:350])  # первые ~350 слов
doc = nlp(snippet)

# Посмотрим 15 токенов с POS/леммой
rows = []
for t in list(doc)[:15]:
    rows.append([t.text, t.lemma_, t.pos_, t.tag_])
pd.DataFrame(rows, columns=["token","lemma","POS","TAG"])


## 5) Простая rule-based система: определение интентов


Сделаем игрушечный **intent detector** для пользовательских сообщений:
- greeting / goodbye
- ask_weather
- ask_time
- math_addition
- other

Подход:
- регулярки + `spaCy Matcher`.


In [ ]:

from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)

pattern_greet = [{"LOWER": {"IN": ["привет","салют", "здаров","здравствуй", "здравствуйте", "приветствую"]}}]
pattern_goodbye = [{"LOWER": {"IN": ["пока", "до свидания", "прощай", "увидимся", "всего доброго"]}}]
pattern_weather = [{"LOWER": {"IN": ["погода", "дождь", "солнечно","сыро", "пасмурно", "ветренно"]}}]
pattern_time = [{"LOWER": {"IN": ["время", "час", "часы", "который час"]}}]

pattern_add_1 = [{"LOWER": "сложи"}, {"IS_DIGIT": True}, {"LOWER": "и"}, {"IS_DIGIT": True}]
pattern_add_2 = [{"LOWER": "сколько"}, {"LOWER": "будет"}, {"IS_DIGIT": True}, {"LOWER": "плюс"}, {"IS_DIGIT": True}]
pattern_add_3 = [{"IS_DIGIT": True}, {"TEXT": "+"}, {"IS_DIGIT": True}]

matcher.add("GREET", [pattern_greet])
matcher.add("GOODBYE", [pattern_goodbye])
matcher.add("WEATHER", [pattern_weather])
matcher.add("TIME", [pattern_time])
matcher.add("ADD", [pattern_add_1, pattern_add_2])

def detect_intent(text: str):
    doc = nlp(text)
    matches = matcher(doc)
    labels = [nlp.vocab.strings[m_id] for m_id, _, _ in matches]

    if "ADD" in labels:
        nums = [int(t.text) for t in doc if t.like_num]
        if len(nums) >= 2:
            return "math_addition", nums[0] + nums[1]
        return "math_addition", None
    if "WEATHER" in labels:
        return "ask_weather", None
    if "TIME" in labels:
        return "ask_time", None
    if "GREET" in labels:
        return "greeting", None
    if "GOODBYE" in labels:
        return "goodbye", None
    return "other", None

tests = [
    "Привет!",
    "Здравствуйте, как дела?",
    "Сложи 34957 и 70764",
    "Сколько будет 2 плюс 2?",
    "15 + 3",
    "Какая погода завтра?",
    "Что там с дождём?",
    "Сколько времени?",
    "Который час?",
    "Пока!",
    "До свидания!",
    "Расскажи анекдот"
]

for s in tests:
    intent, val = detect_intent(s)
    print(f"{s!r} -> {intent}", (f"(value={val})" if val is not None else ""))


Интент 1: "ПОИСК" (поиск информации в интернете)

Интент 2: "НАПОМИНАНИЕ" (установка напоминаний)

In [ ]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)

pattern_search_1 = [{"LOWER": {"IN": ["что такое", "поищи", "найти"]}}, {"LOWER": {"NOT_IN": ["в", "на", "под", "за"]}, "OP": "+"}]
pattern_search_2 = [{"LOWER":{"IN":["где","как"]}}, {"LOWER":{"IN":["найти","узнать"]}}, {"LOWER": {"NOT_IN": ["в", "на", "под", "за"]}, "OP": "+"}]
pattern_search_3 = [{"LOWER": "что"}, {"LOWER": "такое"}, {"LOWER": {"NOT_IN": ["в", "на", "под", "за"]}, "OP": "+"}]
pattern_search_4 = [{"LOWER": "кто"}, {"LOWER": {"IN": ["такой", "такая", "такие"]}}, {"LOWER": {"NOT_IN": ["в", "на", "под", "за"]}, "OP": "+"}]


pattern_reminder_1 = [{"LOWER": "напомни"}, {"LOWER": {"IN": ["мне", "нам"]}, "OP": "?"}, {"LOWER": {"NOT_IN": ["в", "на", "под", "за"]}, "OP": "+"}]
pattern_reminder_2 = [{"LOWER": "напомни"}, {"LOWER": "о"}, {"LOWER": {"NOT_IN": ["в", "на", "под", "за"]}, "OP": "+"}]
pattern_reminder_3 = [{"LOWER": "не"}, {"LOWER": "забудь"}, {"LOWER": {"NOT_IN": ["в", "на", "под", "за"]}, "OP": "+"}]

matcher.add("SEARCH", [pattern_search_1, pattern_search_2, pattern_search_3, pattern_search_4])
matcher.add("REMINDER", [pattern_reminder_1, pattern_reminder_2, pattern_reminder_3])


def detect_intent(text: str):
    doc = nlp(text)
    matches = matcher(doc)
    labels = [nlp.vocab.strings[m_id] for m_id, _, _ in matches]

    if "SEARCH" in labels:
        return "ask_SEARCH", None
    if "REMINDER" in labels:
        return "ask_REMINDER", None
    return "other", None

tests = [
    "что такое нейросеть",
    "поищи рецепт борща",
    "где найти хорошую книгу",
    "как узнать правду",
    "кто такой пушкин",
    "где находится театр",
    "напомни купить хлеб",
    "напомни мне позвонить маме",
    "напомни о встрече",
    "не забудь выключить утюг",
    "запомни этот пароль",
    "сделай заметку купить молоко"
]

for s in tests:
    intent, val = detect_intent(s)
    print(f"{s!r} -> {intent}", (f"(value={val})" if val is not None else ""))


## 8) Лемматизация и стемминг (русский)

Сравним:
- **леммы** (нормальная форма слова) — полезно для словаря/частот;
- **стеммы** (усечённые основы) — грубее, но быстро и просто.


In [ ]:
import re

# Базовая токенизация для русского (буквы + дефис)
ru_tokens = re.findall(r"[А-Яа-яЁё]+(?:-[А-Яа-яЁё]+)?", text.lower())
print("Токены:", ru_tokens)


In [ ]:
# 8.1 Стемминг (Snowball)
import nltk
from nltk.stem.snowball import SnowballStemmer

nltk.download("punkt", quiet=True)
stemmer = SnowballStemmer("russian")

ru_stems = [stemmer.stem(w) for w in ru_tokens]
print(list(zip(ru_tokens, ru_stems)))


### 8.2 Лемматизация через pymorphy2 (обычно лучше для русского)

In [ ]:
!pip -q install pymorphy3 pymorphy3-dicts-ru

In [ ]:
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

ru_lemmas_pym = [morph.parse(w)[0].normal_form for w in ru_tokens]
print(list(zip(ru_tokens, ru_lemmas_pym)))


In [ ]:
import spacy
from spacy import displacy

nlp_ru = spacy.load("ru_core_news_sm")

doc_ru = nlp_ru(text)
print([(t.text, t.lemma_, t.pos_) for t in doc_ru if t.is_alpha])

displacy.render(doc_ru, style="dep", jupyter=True, options={"distance": 90})


### 8.3 Лемматизация через spaCy (удобно, когда вы и так используете пайплайн)

In [ ]:
import spacy
from spacy import displacy

nlp_ru = spacy.load("ru_core_news_sm")

doc_ru = nlp_ru(text)
print([(t.text, t.lemma_, t.pos_) for t in doc_ru if t.is_alpha])

displacy.render(doc_ru, style="dep", jupyter=True, options={"distance": 90})


## 9) POS-теги + синтаксические зависимости (DEP) + NER для русского

In [ ]:
import pandas as pd

# 9.1 POS + морфология
rows = []
for t in doc_ru:
    if t.is_space:
        continue
    rows.append([t.text, t.lemma_, t.pos_, t.tag_, t.morph.to_json()])
pd.DataFrame(rows, columns=["token","lemma","POS","TAG","morph"])


In [ ]:
# 9.2 Синтаксические зависимости (DEP): token -> head
dep_rows = []
for t in doc_ru:
    if t.is_space:
        continue
    dep_rows.append([t.text, t.dep_, t.head.text])
pd.DataFrame(dep_rows, columns=["token","dep","head"])


In [ ]:
# 9.3 NER (именованные сущности)
[(ent.text, ent.label_) for ent in doc_ru.ents]


## Матчеры на основе тэгов

In [ ]:
from spacy.matcher import Matcher
import spacy
nlp = spacy.load("ru_core_news_sm")

matcher = Matcher(nlp.vocab)

pattern_superl_noun = [{"POS": "DET", "OP": "?"}, {"POS": "ADJ", "MORPH": {"IS_SUPERSET": ["Degree=Sup"]}}, {"POS": "NOUN"}]
pattern_superl_russian = [ {"POS": "DET", "OP": "?"},{"POS": "ADJ", "TEXT": {"REGEX": ".+(ейш|айш)(ий|ая|ое|ые)$"}}, {"POS": "NOUN"}]

matcher.add("SUPERL_NOUN_DEGREE", [pattern_superl_noun])
matcher.add("SUPERL_NOUN_REGEX", [pattern_superl_russian])

doc = nlp(text)

print("Прилагательные в тексте:")
for token in doc:
    if token.pos_ == "ADJ":
        print(f"  {token.text}: morph={token.morph}")


In [ ]:
pattern_past_verb_object = [
    {"POS": "VERB", "MORPH": {"IS_SUPERSET": ["Tense=Past"]}},
    {"POS": "DET", "OP": "?"},
    {"POS": {"IN": ["NOUN", "PROPN"]}}
]
matcher.add("PAST_VERB_OBJ", [pattern_past_verb_object])

doc = nlp("а другой день я проснулся с головною болью, смутно припоминая себе вчерашние происшествия. Размышления мои прерваны были Савельичем, вошедшим ко мне с чашкою чая.«Рано, Пётр Андреич, – сказал он мне, качая головою, – рано начинаешь гулять. И в кого ты пошёл? Кажется, ни батюшка, ни дедушка пьяницами не бывали; о матушке и говорить нечего: отроду, кроме квасу, в рот ничего не изволила брать. А кто всему виноват? проклятый мусьё. То и дело, бывало, к Антипьевне забежит: «Мадам, же ву при, водкю». Вот тебе и же ву при! Нечего сказать: добру наставил, собачий сын. И нужно было нанимать в дядьки басурмана, как будто у барина не стало и своих людей!»")
for mid, start, end in matcher(doc):
    print(nlp.vocab.strings[mid], doc[start:end].text)


In [ ]:
pattern_np = [
    {"POS": "DET", "OP": "?"},
    {"POS": "ADJ", "OP": "*"},
    {"POS": {"IN": ["NOUN", "PROPN"]}}
]
matcher.add("NP_RULE", [pattern_np])

doc = nlp("Любопытство меня мучило: куда ж отправляют меня, если уж не в Петербург? Я не сводил глаз с пера батюшкина, которое двигалось довольно медленно. Наконец он кончил, запечатал письмо в одном пакете с паспортом, снял очки и, подозвав меня, сказал: «Вот тебе письмо к Андрею Карловичу P., моему старинному товарищу и другу. Ты едешь в Оренбург служить под его начальством».")
for mid, start, end in matcher(doc):
    print(doc[start:end].text)


In [ ]:
pattern_question = [
    {"POS": "AUX"},
    {"POS": "PRON"},
    {"POS": "VERB"}
]
matcher.add("QUESTION_AUX_PRON_VERB", [pattern_question])

doc = nlp("Вдруг он обратился к матушке: «Авдотья Васильевна, а сколько лет Петруше?»")
for mid, start, end in matcher(doc):
    print(nlp.vocab.strings[mid], doc[start:end].text)


In [ ]:
from spacy.matcher import Matcher
matcher = Matcher(nlp_ru.vocab)

pattern_adj_noun = [
    {"POS": "ADJ"},
    {"POS": "NOUN"}
]
matcher.add("ADJ_NOUN", [pattern_adj_noun])

doc = nlp_ru("Старый дом стоял на тихой улице.")
for mid, start, end in matcher(doc):
    print(nlp_ru.vocab.strings[mid], doc[start:end].text)


## 10) Мини-задача для семинара (русский)

1) Возьмите фрагмент русского романа (например, 30–100k символов). (Сделано)
2) Сравните частоты **по токенам**, **по стеммам**, **по леммам**: (сделано)
   - Как меняются топ-20? стемма и лемма похожи по топ 20 и есть глагол, по токенам топ союзы и местоименнения (сделано)
   - Как меняется доля “длинного хвоста” (≤3) по **types**?
80 процентов используется частые слова, а 20 редкие
3) Выберите 5 предложений и сравните: (сделано)
   - POS/DEP/NER от `spaCy`
   - леммы от `pymorphy3`
4) Составьте список имен персонажей (сделано)
5) Найдите как можно больше предложений, где здороваются (сделано)


In [ ]:
# Простая токенизация для частот: слова из латинских букв + апострофы
tokens = re.findall(r"[А-Яа-я]+(?:'[А-Яа-я]+)?", text.lower())
cnt = Counter(tokens)

N_tokens = sum(cnt.values())  # tokens = все употребления
V_types = len(cnt)            # types  = уникальные слова

print(f"Tokens (всего словоупотреблений): {N_tokens:,}")
print(f"Types  (уникальных слов):        {V_types:,}")

# Топ-20 по токена
top20 = cnt.most_common(20)
top20[:10]


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Zipf данные
freqs_sorted = np.array(sorted(cnt.values(), reverse=True))
ranks = np.arange(1, len(freqs_sorted) + 1)

# 3.1 Zipf log-log
plt.figure(figsize=(7,5))
plt.loglog(ranks, freqs_sorted, marker=".", linestyle="none")
plt.title("Pride and Prejudice: закон Ципфа (ранг–частота)")
plt.xlabel("Ранг слова (log)")
plt.ylabel("Частота (log)")
plt.tight_layout()
plt.show()

# 3.2 Топ-20
df_top20 = pd.DataFrame(top20, columns=["word","count"])
plt.figure(figsize=(10,4))
plt.bar(df_top20["word"], df_top20["count"])
plt.title("Топ-20 слов по частоте")
plt.xlabel("Слово")
plt.ylabel("Частота")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 3.3 Длинный хвост: <= 3
tail_words = [w for w, c in cnt.items() if c <= 3]
tail_types = len(tail_words)
tail_tokens = sum(cnt[w] for w in tail_words)

print(f"Хвост (c<=3): {tail_types} типов = {tail_types/V_types:.1%} словаря")
print(f"Хвост (c<=3): {tail_tokens} токенов = {tail_tokens/N_tokens:.1%} текста")

plt.figure(figsize=(7,4))
plt.bar(["Доля ТИПОВ\n(c ≤ 3)", "Доля ТОКЕНОВ\n(c ≤ 3)"],
        [tail_types/V_types, tail_tokens/N_tokens])
plt.ylim(0, 1)
plt.title("«Длинный хвост» (c ≤ 3)")
plt.ylabel("Доля")
plt.tight_layout()
plt.show()

# 3.4 (Опционально) 10 случайных hapax (c=1)
hapax = [w for w, c in cnt.items() if c == 1]
rng = np.random.default_rng(0)
sample10 = list(rng.choice(hapax, size=10, replace=False))
sample10

In [ ]:
import nltk
from nltk.stem.snowball import SnowballStemmer

nltk.download("punkt", quiet=True)
stemmer = SnowballStemmer("russian")

ru_stems = [stemmer.stem(w) for w in ru_tokens]
cnt = Counter(ru_stems)
top20 = cnt.most_common(20)
top20[:10]


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Zipf данные
freqs_sorted = np.array(sorted(cnt.values(), reverse=True))
ranks = np.arange(1, len(freqs_sorted) + 1)

# 3.1 Zipf log-log
plt.figure(figsize=(7,5))
plt.loglog(ranks, freqs_sorted, marker=".", linestyle="none")
plt.title("Pride and Prejudice: закон Ципфа (ранг–частота)")
plt.xlabel("Ранг слова (log)")
plt.ylabel("Частота (log)")
plt.tight_layout()
plt.show()

# 3.2 Топ-20
df_top20 = pd.DataFrame(top20, columns=["word","count"])
plt.figure(figsize=(10,4))
plt.bar(df_top20["word"], df_top20["count"])
plt.title("Топ-20 слов по частоте")
plt.xlabel("Слово")
plt.ylabel("Частота")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 3.3 Длинный хвост: <= 3
tail_words = [w for w, c in cnt.items() if c <= 3]
tail_types = len(tail_words)
tail_tokens = sum(cnt[w] for w in tail_words)

print(f"Хвост (c<=3): {tail_types} типов = {tail_types/V_types:.1%} словаря")
print(f"Хвост (c<=3): {tail_tokens} токенов = {tail_tokens/N_tokens:.1%} текста")

plt.figure(figsize=(7,4))
plt.bar(["Доля ТИПОВ\n(c ≤ 3)", "Доля ТОКЕНОВ\n(c ≤ 3)"],
        [tail_types/V_types, tail_tokens/N_tokens])
plt.ylim(0, 1)
plt.title("«Длинный хвост» (c ≤ 3)")
plt.ylabel("Доля")
plt.tight_layout()
plt.show()

# 3.4 (Опционально) 10 случайных hapax (c=1)
hapax = [w for w, c in cnt.items() if c == 1]
rng = np.random.default_rng(0)
sample10 = list(rng.choice(hapax, size=10, replace=False))
sample10

In [ ]:
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

ru_lemmas_pym = [morph.parse(w)[0].normal_form for w in ru_tokens]
cnt = Counter(ru_stems)
top20 = cnt.most_common(20)
top20[:10]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Zipf данные
freqs_sorted = np.array(sorted(cnt.values(), reverse=True))
ranks = np.arange(1, len(freqs_sorted) + 1)

# 3.1 Zipf log-log
plt.figure(figsize=(7,5))
plt.loglog(ranks, freqs_sorted, marker=".", linestyle="none")
plt.title("Pride and Prejudice: закон Ципфа (ранг–частота)")
plt.xlabel("Ранг слова (log)")
plt.ylabel("Частота (log)")
plt.tight_layout()
plt.show()

# 3.2 Топ-20
df_top20 = pd.DataFrame(top20, columns=["word","count"])
plt.figure(figsize=(10,4))
plt.bar(df_top20["word"], df_top20["count"])
plt.title("Топ-20 слов по частоте")
plt.xlabel("Слово")
plt.ylabel("Частота")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 3.3 Длинный хвост: <= 3
tail_words = [w for w, c in cnt.items() if c <= 3]
tail_types = len(tail_words)
tail_tokens = sum(cnt[w] for w in tail_words)

print(f"Хвост (c<=3): {tail_types} типов = {tail_types/V_types:.1%} словаря")
print(f"Хвост (c<=3): {tail_tokens} токенов = {tail_tokens/N_tokens:.1%} текста")

plt.figure(figsize=(7,4))
plt.bar(["Доля ТИПОВ\n(c ≤ 3)", "Доля ТОКЕНОВ\n(c ≤ 3)"],
        [tail_types/V_types, tail_tokens/N_tokens])
plt.ylim(0, 1)
plt.title("«Длинный хвост» (c ≤ 3)")
plt.ylabel("Доля")
plt.tight_layout()
plt.show()

# 3.4 (Опционально) 10 случайных hapax (c=1)
hapax = [w for w, c in cnt.items() if c == 1]
rng = np.random.default_rng(0)
sample10 = list(rng.choice(hapax, size=10, replace=False))
sample10

In [ ]:
import spacy
import pymorphy3

In [ ]:
nlp = spacy.load("ru_core_news_sm")
morph = pymorphy3.MorphAnalyzer()

In [ ]:
sentences = [
    "Красивая девушка медленно шла по осеннему парку.",
    "Вчера в Москве состоялась премьера нового фильма.",
    "Старый пёс громко залаял на незнакомого почтальона.",
    "Петр Сергеевич купил в магазине большую и вкусную клубнику.",
    "Интересная книга лежала на столе в моей комнате."
]

In [ ]:
for i, sent in enumerate(sentences, 1):
    print("\nspaCy:")
    doc = nlp(sent)
    for token in doc:
        ner = token.ent_type_ if token.ent_type_ else "_"
        print(f"  {token.text:15} POS: {token.pos_:10} DEP: {token.dep_:10} NER: {ner}")
    print("\npymorphy3 (леммы):")
    words = sent.replace(".", "").split()
    for word in words:
        parse = morph.parse(word)[0]
        print(f"  {word:15} -> {parse.normal_form}")